# Baseline

**Autor:** jd · **Datum:** 2026-09-30 · **Issue:** #27

**Ziel:** Referenzwerte für Meilenstein 1, gegen die alle späteren Ansätze antreten. Bewusst
**kein** begründeter Ansatz (siehe #11/#54): eine Untergrenze (Dummy) und ein einfaches Modell,
das Crop und Stage nativ gemeinsam vorhersagt (Random Forest). Ablauf wie in
`docs/modelle/ansatz-entwickeln.md`: `tune()` auf den CV-Folds, dann einmal `run()`.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier

from awp2.config import SEED
from awp2.evaluation import plot_confusion_matrices
from awp2.experiment import RunConfig, TuneConfig, run, tune

## 1. Untergrenze: immer die häufigste Klasse

In [ ]:
dummy = run(
    DummyClassifier(strategy="most_frequent"),
    RunConfig(
        name="dummy_most_frequent",
        approach="baseline",
        description="Untergrenze: immer die häufigste Kultur und das häufigste Stadium.",
    ),
)
dummy.metrics.model_dump()

## 2. Random Forest: Einstellungen per Cross-Validation suchen

Nur auf dem Train-Teil (5 Folds). Kleines Raster: Tiefe, Blattgröße, Anteil der Features je Split.

In [ ]:
tuned = tune(
    RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1),
    TuneConfig(
        name="rf_search",
        approach="baseline",
        description="Random Forest (Crop und Stage gemeinsam): Tiefe, Blattgröße, max_features.",
        param_grid={
            "max_depth": [None, 20],
            "min_samples_leaf": [1, 3],
            "max_features": ["sqrt", 0.3],
        },
    ),
)
tuned.candidates.sort_values("rank")[["params", "mean", "std"]]

## 3. Einmal auf der Validierung bewerten

In [ ]:
rf = run(
    tuned.best_model,
    RunConfig(
        name="rf_baseline",
        approach="baseline",
        description="Bester Random Forest aus rf_search, einmal auf der Validierung bewertet.",
        tuning_run=tuned.run_id,
    ),
)
pd.DataFrame(
    {"Dummy": dummy.metrics.model_dump(), "Random Forest": rf.metrics.model_dump()}
).round(3)

In [ ]:
plot_confusion_matrices(rf.y_val, rf.y_pred);

## Ergebnis

| | BAcc Crop | BAcc Stage | BAcc kombiniert |
| --- | --- | --- | --- |
| Dummy | 0.200 | 0.167 | 0.044 |
| Random Forest | 0.890 | 0.870 | 0.743 |

- Beste Einstellung per CV: `max_depth=20`, `max_features=0.3`, `min_samples_leaf=1`
  (CV 0.750 ± 0.019, Validierung 0.743 – kein Hinweis auf Überanpassung).
- Verwechslungen: Mais ↔ Soja; benachbarte Stadien (Emerge_VEarly → Early_Mid,
  Critical → Mature_Senesc).
- 0,3 % unmögliche Kultur/Stadium-Paare → Argument für hierarchische/kombinierte Ansätze.

Details: `docs/modelle/ansaetze.md` (Baseline), MLflow-Experiment `crop-stage`.